# ffn_cellsim DCM — σ × T1-rate aggregation factorial (Colab GPU)

Native **N=400 full-compartment** DCM spheroid. Tests the 2×2 factorial
**σ (aggregate Foty-Steinberg tension) × T1-rate (KMC tissue rearrangement)** —
does rate/event T1 rearrangement let σ-compaction continue *past* the mechanical
saturation (~100 s) toward biology-time? (σ = END-STATE mechanical packing; T1 = the
slow rearrangement large-dt mechanics correctly freeze.)

**Fastest path (least wall-time):** duplicate this notebook **once per condition**
(4 Colab GPUs), set a different `COND` in each, Run all → the 4 finish in ~one
condition's time (**~1 h on L4/A100, ~2.8 h on T4**). Or `COND='all'` = sequential in one session.

Native N=400 ≈ 5 steps/s on an A5000; **pick a Pro GPU (L4/A100)** — a free T4 is ~0.5× and would be slower than gbook.

In [ ]:
# 1. confirm the Colab GPU  (Runtime ▸ Change runtime type ▸ GPU;  prefer L4 / A100)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

In [ ]:
# 2. clone the repo + checkout the σ×T1 branch (public repo → no token needed)
import os
TOKEN  = ''                          # public repo → leave ''
BRANCH = 'dcm/sigma-t1-colab'
auth = f'{TOKEN}@' if TOKEN else ''
url  = f'https://{auth}github.com/sungwook9997/ffn_cellsim.git'
if not os.path.isdir('ffn_cellsim'):
    !git clone --branch {BRANCH} --depth 1 {url} ffn_cellsim
%cd ffn_cellsim
!git log --oneline -1

In [ ]:
# 3. install Warp (Colab already has numpy / scipy / matplotlib)
!pip -q install warp-lang==1.14.0
import warp as wp; wp.init()   # JIT-compiles the CUDA kernels on first use

In [ ]:
# 4. SSH tunnel (cloudflared) — lets Claude DRIVE the run over SSH on this runtime.
#    Run this cell, then paste the printed HOST to Claude. Keep this session/tab running
#    (the tunnel + the sim live only while the Colab runtime is alive).
import os, subprocess, time, re, urllib.request

CLAUDE_PUBKEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIEfrdXO15uVuO0Qx2xhWgYW6yWNZZfAH2OQGSAu0cLQ9 claude-colab-ffn"

# authorize Claude's key for root (public key — safe to embed; only the private key grants access)
os.makedirs('/root/.ssh', exist_ok=True)
open('/root/.ssh/authorized_keys','w').write(CLAUDE_PUBKEY + "\n")
os.chmod('/root/.ssh', 0o700); os.chmod('/root/.ssh/authorized_keys', 0o600)

# sshd: key-only root login
os.system("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openssh-server >/dev/null 2>&1")
os.makedirs('/run/sshd', exist_ok=True)
os.system("ssh-keygen -A >/dev/null 2>&1")
os.system(r"sed -i 's/#\?PermitRootLogin.*/PermitRootLogin prohibit-password/' /etc/ssh/sshd_config")
os.system(r"sed -i 's/#\?PubkeyAuthentication.*/PubkeyAuthentication yes/' /etc/ssh/sshd_config")
os.system("pkill sshd 2>/dev/null; /usr/sbin/sshd")

# cloudflared quick tunnel over ssh
if not os.path.exists('/usr/local/bin/cloudflared'):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/usr/local/bin/cloudflared'); os.chmod('/usr/local/bin/cloudflared', 0o755)
os.system("pkill cloudflared 2>/dev/null")
subprocess.Popen("/usr/local/bin/cloudflared tunnel --no-autoupdate --url ssh://localhost:22 "
                 "> /content/cf.log 2>&1", shell=True)

host=None
for _ in range(45):
    time.sleep(2)
    try:
        m=re.search(r'https://([-\w]+\.trycloudflare\.com)', open('/content/cf.log').read())
        if m: host=m.group(1); break
    except FileNotFoundError: pass
print("="*64)
print("SSH HOST :", host or "(not ready — wait 10s and re-run this cell)")
print("Repo is at /content/ffn_cellsim ; Warp installed. Paste the HOST to Claude.")
print("="*64)

In [ ]:
# 5. (FALLBACK if not driving via SSH) σ × T1-rate factorial — native N=400 full-compartment, BDF2, accel_dt=8e-3.
#    Set COND to ONE condition and DUPLICATE this notebook per GPU for 4× parallelism,
#    or COND='all' to run the four sequentially in one session.
import os, subprocess, time
os.environ['PYTHONPATH'] = '.'
OUT = '/content/out'; os.makedirs(OUT, exist_ok=True)

STEPS = '30000'      # 240 s biology-time @ accel_dt=8e-3 (same step-budget as the T1 HERO)
COND  = 'all'        # <-- 'c00_base' | 'c10_sig' | 'c01_t1' | 'c11_both' | 'all'

CONDS = {
    'c00_base': dict(SIGMA='0',      T1_RATE='0'),   # loose, no driver           (negative control)
    'c10_sig' : dict(SIGMA='5.0e-3', T1_RATE='0'),   # mechanical σ-compaction     (plateaus ~100 s)
    'c01_t1'  : dict(SIGMA='0',      T1_RATE='1'),   # T1 fluidisation, no σ drive
    'c11_both': dict(SIGMA='5.0e-3', T1_RATE='1'),   # σ + T1                       (the hypothesis)
}
COMMON = dict(NCELLS='400', WARMUP='500', STEPS=STEPS, ACCEL_DT='8e-3', GAP='2.4',
              BUNDLE='10', ENUC='399', RNUC='0.7', GAMMA='5e-4', NUCLEUS='1', FRAMES='40',
              BDF2='1', T1_MODE='rigid', T1_CADENCE='200', T1_SEED='13',
              DEVICE='cuda:0', OUT_DIR=OUT)

for name in (list(CONDS) if COND == 'all' else [COND]):
    env = {**os.environ, **COMMON, **CONDS[name], 'TAG': f'_{name}'}
    print(f'\n===== RUN {name}: SIGMA={CONDS[name]["SIGMA"]} T1_RATE={CONDS[name]["T1_RATE"]} =====', flush=True)
    t0 = time.time()
    logp = f'{OUT}/fac_{name}.log'
    with open(logp, 'w') as log:
        p = subprocess.Popen(['python', 'ffn_sim/scripts/_gbook_aggregate_compaction.py'],
                             env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:            # stream live to the notebook AND tee to a log
            print(line, end=''); log.write(line); log.flush()
        p.wait()
    print(f'===== {name} DONE in {(time.time()-t0)/60:.1f} min =====', flush=True)

In [ ]:
# 6. (FALLBACK) collect the per-condition SUMMARY + zip the logs (+ npz) for download
import glob, os, subprocess
OUT = '/content/out'
print('==== FACTORIAL SUMMARIES ====')
for log in sorted(glob.glob(f'{OUT}/fac_*.log')):
    for line in open(log):
        if '[SUMMARY' in line:
            print(os.path.basename(log)[4:-4], '|', line.strip())
subprocess.run(['zip','-jq','/content/sigma_t1_factorial.zip',
                *glob.glob(f'{OUT}/fac_*.log'), *glob.glob(f'{OUT}/agg_compaction_n400_*.npz')])
print('\nzip MB:', round(os.path.getsize('/content/sigma_t1_factorial.zip')/1e6, 1))
from google.colab import files
files.download('/content/sigma_t1_factorial.zip')